In [1]:
import os
if 'google.colab' in str(get_ipython()):
    # Running in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks"
else:
    # Running locally (Mac/Linux)<<
    PROJECT_DIR = '/Users/erikdalgard/Library/CloudStorage/GoogleDrive-dalgard.erik@gmail.com/My Drive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks'

os.chdir(PROJECT_DIR)


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tqdm
from tensorflow import keras
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger, BackupAndRestore
from sklearn.utils.class_weight import compute_class_weight
from datetime import datetime
import glob
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from cub_cutter_c import LunaDataset, make_tf_dataset, training_validation_split
from tqdm.auto import tqdm
from froc_eval import evaluate, bootstrap_cpm, plot_froc, compute_froc, froc_table


#Importing modules
import models_2D, models_3D, models_ae

#Changing so keras trains with float16 instead of float32 to increase computational speed.
#mixed_precision.set_global_policy('mixed_float16')

In [3]:
#Getting 2D CNN models
archi_1_2D = models_2D.get_archi_1_2D()
archi_2_2D = models_2D.get_archi_2_2D()
archi_3_2D = models_2D.get_archi_3_2D()

#Getting 3D CNN models
archi_1_3D = models_3D.get_archi_1_3D()
archi_2_3D = models_3D.get_archi_2_3D()
archi_3_3D = models_3D.get_archi_3_3D()

#Getting autoencoder models
archi_1_ae = models_ae.get_archi_1_AE()
archi_2_ae = models_ae.get_archi_2_AE()
archi_3_ae = models_ae.get_archi_3_AE()

In [10]:
def train_network(model, X_train, y_train=None, X_val=None, y_val=None, project_dir="", epochs=50, batch_size=64, is_ae=False):
    """
    Compiles, logs, and trains a given Keras model. Supports both standard classification
    and Autoencoder (AE) reconstruction training.

    Parameters:
        model (keras.Model): The uncompiled Keras model to be trained.
        X_train: Training features.
        y_train: Training labels (Ignored if is_ae=True).
        X_val: Validation features (Optional).
        y_val: Validation labels (Ignored if is_ae=True).
        project_dir (str): Directory for saving assets.
        epochs (int): Maximum number of training iterations.
        batch_size (int): Number of samples per training batch.
        is_ae (bool): Set to True if training an autoencoder.
        is_debug (bool): If set to True, only run 3 epochs on a small set of the data. No data/model is saved

    Returns:
        History: The history as a keras object
        checkpoint_path: The file path to the weights of the model
        log_path: The file path to the csv logs of the training


    """
    if is_ae:
        loss_function = "mse"
        metrics_list = None
        monitor_metric = "val_loss"
        monitor_mode = "min"
    else:
        loss_function = "binary_crossentropy"
        metrics_list = [
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="sensitivity"),
            keras.metrics.AUC(name="auc"),
        ]       
        monitor_metric = "val_auc"
        monitor_mode = "max"

    # 2. Compile the model with the chosen settings and AdamW as optimizer
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=1e-4,
            weight_decay=1e-4
        ),
        loss=loss_function,
        metrics=metrics_list
    )
    # 3. Create a clean subfolder dynamically named after the architecture
    model_folder = os.path.join(project_dir, 'training_history')
    history_folder = os.path.join(model_folder, model.name)
    os.makedirs(model_folder, exist_ok=True)
    os.makedirs(history_folder, exist_ok=True)

    time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_path = os.path.join(history_folder, f'best_model_{time_stamp}.keras')
    log_path = os.path.join(history_folder, f'training_log_{time_stamp}.csv')

    # 4. Setting up automation callbacks
    callbacks = [
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor_metric,
            mode=monitor_mode,
            save_best_only=True,
            save_weights_only=False  # Explicitly set this to prevent default leaks
        ),
        EarlyStopping( #If model does not improve after 10 epochs, we stop training and restore best model
            monitor=monitor_metric,
            mode=monitor_mode,
            patience=5,
            restore_best_weights=True,
        ),
        CSVLogger(log_path), #logs the training

        ReduceLROnPlateau( #If we stop learning, we wait 5 epochs and then half the learning rate
            monitor=monitor_metric,
            mode=monitor_mode,
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        BackupAndRestore( #If google collabe crashes, this creates a backup folder with the weights of the previous training. If training is succeeded the backup file is deleted.
            backup_dir=os.path.join(history_folder, 'backup')
    ),
    ]


    # 5. Execute Training
    print(f"Launching training loop for: {model.name}")

    history = model.fit(
        X_train,
        validation_data=X_val,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )


    print(f"\nSuccessfully finished training {model.name}!")
    print(f"-> Best weights secured at: {checkpoint_path}")
    print(f"-> History saved to: {log_path}")

    return history, checkpoint_path, log_path

In [ ]:
## OBSERVE THIS IS THE OLD WAY DO NOT USE ANYMORE. Disk → load ALL patches → giant NumPy array in RAM → feed to model. RAM CRASHES
def cut_cubes(arch, layout, subset, is_test=False):
    SUBSET = Path(PROJECT_DIR) / f"data/{subset}"
    VAL_FRACTION = 0.2
    NEG_RATIO = 1
    SEED = 0

    luna = LunaDataset(SUBSET, neg_ratio=NEG_RATIO, seed=SEED, arch=arch, layout=layout)

    feat_shape = tuple(luna.output_signature()[0].shape)
    all_indices = np.arange(len(luna.samples))

    print("feat_shape   :", feat_shape)
    print("total samples:", len(luna.samples))

    def materialize(idxs, desc="Loading samples"):
        X = np.empty((len(idxs), *feat_shape), dtype=np.float32)
        y = np.empty((len(idxs), 1),           dtype=np.float32)
        for k, i in enumerate(tqdm(idxs, desc=desc)):
            X[k], y[k] = luna.get_sample(int(i))
        return X, y

    # --- test mode: no split, return everything as one set ---
    if is_test:
        estimated_gb = (len(all_indices) * np.prod(feat_shape) * 4) / (1024**3)
        print(f"TEST MODE — no train/val split")
        print(f"Estimated size: {estimated_gb:.2f} GB")

        X_test, y_test = materialize(all_indices, desc="Loading test samples")

        print(f"X_test {X_test.shape}  y_test {y_test.shape}  pos {int(y_test.sum())}")
        return X_test, y_test

    # --- normal mode: scan-level train/val split ---
    sample_uids = np.array([
        (luna.pos if label == 1 else luna.neg).iloc[row_i]["seriesuid"]
        for label, row_i, *_ in luna.samples
    ])
    unique  = np.unique(sample_uids)
    n_val   = max(1, int(round(VAL_FRACTION * len(unique))))
    val_uids = set(np.random.default_rng(SEED).choice(unique, size=n_val, replace=False))
    is_val  = np.array([u in val_uids for u in sample_uids])

    estimated_gb = (np.where(~is_val)[0].shape[0] * np.prod(feat_shape) * 4) / (1024**3)
    print(f"Estimated X_train size: {estimated_gb:.2f} GB")

    X_train, y_train = materialize(np.where(~is_val)[0], desc="Loading train samples")
    X_val,   y_val   = materialize(np.where( is_val)[0], desc="Loading val samples")

    print(f"X_train {X_train.shape}  y_train {y_train.shape}  pos {int(y_train.sum())}")
    print(f"X_val   {X_val.shape}    y_val   {y_val.shape}    pos {int(y_val.sum())}")
    return X_train, y_train, X_val, y_val

In [ ]:
#OLD TRAINING SEQUENCE
history, ckpt, log = train_network(
    model=archi_1_3D,          # match ARCH/LAYOUT above
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    project_dir=PROJECT_DIR,
    epochs=50, batch_size=64,
    is_ae=False, is_debug=False,
)


In [8]:

#Getting the training and validation splits
train_tfds, val_tfds = training_validation_split(PROJECT_DIR, "archi2", "3d")

In [9]:
#Training the network
train_network(archi_2_3D, X_train=train_tfds, X_val=val_tfds, project_dir=PROJECT_DIR)

NameError: name 'train_network' is not defined

In [ ]:
#FULL TRAINING LOOP
model_list = [
    ("archi1", "2d", archi_1_2D),
    ("archi2", "2d", archi_2_2D),
    ("archi3", "2d", archi_3_2D),
    ("archi1", "3d", archi_1_3D),
    ("archi2", "3d", archi_2_3D),
    ("archi3", "3d", archi_3_3D),
]

for arch, layout, model in model_list:
    train_tfds, val_tfds = training_validation_split(PROJECT_DIR, arch, layout)
    train_network(model, X_train=train_tfds, X_val=val_tfds, project_dir=PROJECT_DIR)

In [4]:
#LOADING ALL MODELS AFTER TRAINING
model_paths_2D = glob.glob("training_history/2D*/best_model_*.keras", recursive=True)
model_paths_3D = glob.glob("training_history/3D*/best_model_*.keras", recursive=True)
model_paths_AE = glob.glob("training_history/AE*/best_model*.keras", recursive=True)

models_paths_combined = [("archi1", "2d", model_paths_2D[2]),
                         ("archi2", "2d", model_paths_2D[1]),
                         ("archi3", "2d", model_paths_2D[0]),
                         ("archi1", "3d", model_paths_2D[2]),
                         ("archi2", "3d", model_paths_2D[1]),
                         ("archi3", "3d", model_paths_3D[0]),
                         ]

training_paths_2D = glob.glob("training_history/2D*/training_log*.csv", recursive=True)
training_paths_3D = glob.glob("training_history/3D*/training_log*.csv", recursive=True)
training_paths_AE = glob.glob("training_history/AE*/training_log*.csv", recursive=True)


In [ ]:
subset_dir = Path(PROJECT_DIR) / "data/masked_scans3"

all_results = {}

for arch, layout, model_path in models_paths_combined:
    name = f"{arch}_{layout}"
    print(f"Evaluating {name}")
    model = keras.models.load_model(model_path)
    all_results[name] = evaluate(model, subset_dir, arch, layout)
    keras.backend.clear_session()

#Doing the linear combination of the family architectures
w1, w2, w3 = 0.3, 0.4, 0.3
w_total = w1 + w2 + w3

res1 = all_results["archi1_2d"]
res2 = all_results["archi2_2d"]
res3 = all_results["archi3_2d"]

ensemble_probs = (w1 * res1["probs"] + w2 * res2["probs"] + w3*res3['probs']) / w_total
res_ensemble   = compute_froc(ensemble_probs, res1["labels"], res1["n_scans"])

fig, ax = plt.subplots(figsize=(7, 5))
plot_froc(res1, ax=ax, label=f"archi1      (CPM={res1['cpm']:.4f})")
plot_froc(res2, ax=ax, label=f"archi2      (CPM={res2['cpm']:.4f})")
plot_froc(res2, ax=ax, label=f"archi3      (CPM={res2['cpm']:.4f})")
plot_froc(res_ensemble, ax=ax, label=f"Ensemble  (CPM={res_ensemble['cpm']:.4f})")
ax.set_title(f"FROC comparison — 2D CNN models")
plt.tight_layout()
plt.show()

Evaluating archi1_2d
594/594 [==============================] - 30s 50ms/step


Evaluating archi2_2d
594/594 [==============================] - 39s 64ms/step
Evaluating archi3_2d


594/594 [==============================] - 87s 147ms/step


KeyError: 'archi1'

In [ ]:
#TESTING ALL MODELS ON TEST SET

subset_dir = Path(PROJECT_DIR) / "data/masked_scans3"

all_results = {}

#Evaluating the test sets for each model and saving the results in the all_results list
for arch, layout, model_path in models_paths_combined:
    name = f"{arch}_{layout}"
    print(f"Evaluating {name}")
    model = keras.models.load_model(model_path)
    all_results[name] = evaluate(model, subset_dir, arch, layout)
    keras.backend.clear_session()

#Doing the linear combination of the family architectures
w1, w2, w3 = 0.3, 0.4, 0.3
w_total = w1 + w2 + w3

families = {
    "2D": ("archi1_2d", "archi2_2d"),
    "3D": ("archi1_3d", "archi2_3d"),
}

#For each family architecture we take the result and combine it with the weights. Then plot all in the same plot.
for family_name, (key1, key2) in families.items():
    print(f"\n{'='*50}")
    print(f"{family_name} family")
    print(f"{'='*50}")

    res1 = all_results[key1]
    res2 = all_results[key2]

    ensemble_probs = (w1 * res1["probs"] + w2 * res2["probs"]) / w_total
    res_ensemble   = compute_froc(ensemble_probs, res1["labels"], res1["n_scans"])

    fig, ax = plt.subplots(figsize=(7, 5))
    plot_froc(res1, ax=ax, label=f"{key1}      (CPM={res1['cpm']:.4f})")
    plot_froc(res2, ax=ax, label=f"{key2}      (CPM={res2['cpm']:.4f})")
    plot_froc(res_ensemble, ax=ax, label=f"Ensemble  (CPM={res_ensemble['cpm']:.4f})")
    ax.set_title(f"FROC comparison — {family_name} models")
    plt.tight_layout()
    plt.show()


In [ ]:
#TESTING ALL MODELS ON TEST SET

#Test subset
subset_dir = Path(PROJECT_DIR) / "data/masked_scans3"

#Loading the models
archi_2_2D, archi_1_2D  = keras.models.load_model(model_paths_3D[0]), keras.models.load_model(model_paths_3D[1])

#Assigning weights for linear combination
w1, w2 = 0.43, 0.57
w_total = w1 + w2


#Evaluating the two models
print("Evaluating archi1 2D...")
res1_2D = evaluate(archi_1_2D, subset_dir, arch="archi1", layout="3d")

print("Evaluating archi2 2D...")
res2_2D = evaluate(archi_2_2D, subset_dir, arch="archi2", layout="3d")


ensemble_probs = (w1 * res1_2D["probs"] + w2 * res2_2D["probs"]) / w_total
res_ensemble = compute_froc(ensemble_probs, res1_2D["labels"], res1_2D["n_scans"])

# --- combined plot ---
fig, ax = plt.subplots(figsize=(7, 5))
plot_froc(res1_2D,    ax=ax, label=f"archi1    (CPM={res1_2D['cpm']:.4f})")
plot_froc(res2_2D,    ax=ax, label=f"archi2    (CPM={res2_2D['cpm']:.4f})")
plot_froc(res_ensemble, ax=ax, label=f"Ensemble  (CPM={res_ensemble['cpm']:.4f})")
ax.set_title("FROC comparison — 3D models")
plt.tight_layout()
plt.show()

table = froc_table(
    ("Archi-1",  res1_2D),
    ("Archi-2",  res2_2D),
    ("Ensemble", res_ensemble),
)
print(table)